# 3dmodel_gen — Remote GPU server (Colab)

Runs the `m3d_ai` FastAPI server with the **TripoSR** adapter, exposed via **ngrok** so a local desktop app on a non-GPU laptop can call it.

**Before you run:**
1. Runtime → Change runtime type → **GPU (T4)**.
2. Drop your ngrok auth token into Colab Secrets as `NGROK_AUTHTOKEN` (free tier is fine).
3. (Optional) Drop a HuggingFace token as `HF_TOKEN` if you ever switch to a gated model.
4. Set `BACKEND_TOKEN` to a random secret — paste the same value into the desktop app's `M3D_GPU_BACKEND_TOKEN`.

When the last cell prints `https://xxxx.ngrok-free.app`, paste that into the app's Settings → GPU backend URL and switch backend to `remote`.

Sessions time out after ~12 h. Per docs/RESUMABILITY_AND_BUDGET.md, in-flight jobs pause and resume cleanly when the URL comes back.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Clone the repo + install deps

Replace `REPO_URL` with your fork. Keep the commit pinned so this notebook is reproducible.

In [ ]:
REPO_URL = 'https://github.com/YOUR_USER/3dmodel_gen.git'  # <- edit
REPO_REF = 'main'                                            # <- edit; pin a commit when you've stabilized

import os, subprocess, sys
if not os.path.isdir('/content/3dmodel_gen'):
    subprocess.check_call(['git', 'clone', REPO_URL, '/content/3dmodel_gen'])
%cd /content/3dmodel_gen
!git fetch origin && git checkout $REPO_REF
print('HEAD:', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())

In [ ]:
# Install m3d_ai + the torch extra. Colab already has torch; we let pip resolve.
%cd /content/3dmodel_gen/ai_models
!pip install -q -e '.[torch,dev]'

# TripoSR is a github install (not on PyPI). Pin the commit listed in registry.yaml.
TRIPOSR_REV = 'ea0080e5559a26ce81c1d34d869fda37c4f3e9c7'
!pip install -q 'git+https://github.com/VAST-AI-Research/TripoSR.git@'$TRIPOSR_REV

# Sanity import
import importlib
importlib.import_module('tsr.system')
print('TripoSR OK')

## 3. Configure the server

Set the bearer token + adapter. The token is what the desktop app will send in `Authorization: Bearer ...`.

In [ ]:
from google.colab import userdata
import os, secrets

# Token: either provided in Colab Secrets, or auto-generated and printed.
try:
    backend_token = userdata.get('BACKEND_TOKEN')
except Exception:
    backend_token = None
if not backend_token:
    backend_token = secrets.token_urlsafe(24)
    print('Generated BACKEND_TOKEN (also paste into desktop app):')
    print(backend_token)

os.environ['M3D_AI_AUTH_TOKEN'] = backend_token
os.environ['M3D_AI_PRIMARY_GENERATOR'] = 'triposr'
os.environ['M3D_AI_PORT'] = '8000'
os.environ['HF_HOME'] = '/content/models_cache'
os.environ['TRANSFORMERS_OFFLINE'] = '0'

# Optional HF token for gated models (Hunyuan in M4).
try:
    hf = userdata.get('HF_TOKEN')
    if hf:
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf
        print('HF token loaded.')
except Exception:
    pass

## 4. Start the server in the background

In [ ]:
import subprocess, time, signal, os

%cd /content/3dmodel_gen/ai_models
log_path = '/content/server.log'
log = open(log_path, 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'm3d_ai.remote_server:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
    stdout=log, stderr=subprocess.STDOUT,
    env={**os.environ},
)
print('Server PID:', server.pid)
# Give it a moment to warm up the adapter (downloads TripoSR weights on first run).
time.sleep(8)
!tail -n 40 /content/server.log

In [ ]:
# Wait for /health to succeed (TripoSR weights download ~1 GB on first run).
import urllib.request, time, json
for i in range(60):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3) as r:
            print(json.loads(r.read()))
            break
    except Exception as e:
        print(f'[{i}] waiting:', e)
        time.sleep(5)
else:
    raise RuntimeError('server did not come up')

## 5. Expose via ngrok

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok, conf
from google.colab import userdata

auth = userdata.get('NGROK_AUTHTOKEN')
if not auth:
    raise RuntimeError('Set NGROK_AUTHTOKEN in Colab Secrets first')
ngrok.set_auth_token(auth)

public = ngrok.connect(8000, proto='http')
print()
print('=' * 60)
print('Paste this into the desktop app  (Settings → GPU backend URL):')
print()
print('  ', public.public_url)
print()
print('Bearer token (Settings → GPU backend token):')
print('  ', os.environ['M3D_AI_AUTH_TOKEN'])
print('=' * 60)

## 6. Tail the server log (leave this running)

Interrupt to stop. The ngrok tunnel + server stay up until you shut down the Colab kernel.

In [ ]:
!tail -F /content/server.log